In [ ]:
import pyspark.sql.functions as F
import requests

In [0]:
#── Unity Catalog location (must match the other scripts in this pipeline) ──
# See the README's "Key concepts" section for what catalog/schema mean.
CATALOG = "use1_prod_artemis_catalog_3718194974443840" #change
SCHEMA = "tier1_raw" #change
 
# ── Source tables produced by earlier jobs ──────────────────────────────────
flights_table = f"{CATALOG}.{SCHEMA}.drone_mission_table"  # from Job 1 (1_Flights_update)
ortho_table = f"{CATALOG}.{SCHEMA}.drone_ortho_table"       # from 2_update_tif
 
flights_df = spark.table(flights_table)
raw_ortho_df = spark.table(ortho_table)
 
# ── Step 1: Which flights already have their orthomosaic generated? ────────
# If the "ortho_exists" column is present, only count flights where it's True
# as actually finished. If the column isn't there yet (e.g. first run), treat
# the whole table as the "finished" list as-is.
if "ortho_exists" in raw_ortho_df.columns:
    finish_df = raw_ortho_df.filter(F.col("ortho_exists") == True)
else:
    finish_df = raw_ortho_df
 
# ── Step 2: Find the flights that still need an orthomosaic ────────────────
# A "left anti join" keeps only flights from flights_df that have NO match in
# finish_df — i.e. flights that haven't had their orthomosaic built yet.
missing_df = flights_df.join(
    finish_df,
    on=['site', 'trial', 'season', 'flight_date'],
    how='left_anti'
)
 
# ── Step 3: Report the current status ───────────────────────────────────────
total_flights = flights_df.count()
total_finish = finish_df.count()
total_missing = missing_df.count()
 
print("-" * 40)
print(f" INVENTORY REPORT:")
print(f"Total flights detected: {total_flights}")
print(f"Orthomosaics physically generated: {total_finish}")
print(f"Flights pending processing: {total_missing}")
print("-" * 40)
 
if total_missing > 0:
    display(missing_df)
else:
    print(" Everything is up to date! There are no pending flights.")

----------------------------------------
 INVENTORY REPORT:
Total flights detected: 6
Orthomosaics already generated: 6
 Flights pending processing: 0
----------------------------------------
 Everything is up to date! There are no pending flights.


In [0]:
# If there's pending work, trigger the heavy Agisoft job ─────────
# Unlike the other orchestrators in this pipeline (which set task values for
# a downstream task in the SAME job), this one directly calls the Databricks
# REST API to trigger a DIFFERENT job — Job 2 (PIPELINE_ORTHOMOSAIC), which
# runs on the Agisoft Metashape cluster.
if total_missing > 0:
    print(f" Green flag: {total_missing} pending flights. Triggering ONE heavy job...")
 
    JOB_2_ID = 975722269459594 #change  -- Databricks Job ID for PIPELINE_ORTHOMOSAIC
 
    # Get the current workspace host + auth token so we can call the REST API
    # without hardcoding credentials.
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host = ctx.apiUrl().get()
    token = ctx.apiToken().get()
 
    url = f"{host}/api/2.1/jobs/run-now"
    headers = {"Authorization": f"Bearer {token}"}
 
    # 1. We extract ALL the paths into a Python list
    flights_to_process = [row['flight_metadata_path'] for row in missing_df.select('flight_metadata_path').collect()]
 
    # 2. We convert that list into a single comma-separated String.
    # This is necessary because notebook_params only accepts string values,
    # so all the pending flight paths are packed into one parameter.
    mixed_paths = ",".join(flights_to_process)
 
    # 3. We trigger a SINGLE "Run" by passing all the routes in the parameter.
    # This means ALL pending flights are processed in one Agisoft run, rather
    # than triggering a separate job run per flight.
    data = {
        "job_id": JOB_2_ID,
        "notebook_params": {
            # We changed the parameter name to plural as a good practice
            "flight_metadata_paths": mixed_paths
        }
    }
 
    response = requests.post(url, headers=headers, json=data)
 
    if response.status_code == 200:
        # A 200 response means Databricks accepted the request and started the run.
        run_id = response.json().get('run_id', 'Unknown')
        print(f" ->  Trigger successful! (Run ID: {run_id})")
        print(f" -> Were sent {len(flights_to_process)} missions to the heavy cluster in a single block.")
    else:
        # Anything else means the trigger failed — print the API's error
        # message so it's clear what went wrong (e.g. wrong job ID, permissions issue).
        print(f" ->  [ERROR] triggering Job: {response.text}")
 
else:
    # No pending flights — nothing to trigger, so the Agisoft cluster stays off.
    print("  Red flag : There are no pending flights. Heavy cluster will remain off.")

 There are no pending flights..
